In [ ]:
from google.colab import drive

drive.mount('/content/drive')

!pip install ultralytics split-folders

In [2]:
!mkdir -p /content/dataset/input
!unzip -q /content/drive/MyDrive/Train_YOLO/dataset.zip -d /content/dataset/input

In [ ]:
import os
import shutil
import time

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import splitfolders
from sklearn.metrics import classification_report, confusion_matrix
from ultralytics import YOLO

In [ ]:
dataset_input = r"./dataset/input"
dataset_output = r"./dataset/output"
drive_save_path = "/content/drive/MyDrive/Train_YOLO"


def format_duration(total_seconds: float) -> str:
    total_seconds = max(0, int(total_seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


if __name__ == "__main__":
    if os.path.exists(dataset_output):
        shutil.rmtree(dataset_output)

    splitfolders.ratio(
        dataset_input, output=dataset_output, seed=1303, ratio=(0.70, 0.20, 0.10)
    )

    # Khởi tạo Model và Huấn luyện
    model = YOLO("yolov8l-cls.pt")
    start_time = time.perf_counter()

    results = model.train(
        data=dataset_output,
        epochs=120,
        imgsz=384,
        batch=64,
        optimizer="SGD",
        momentum=0.937,
        patience=20,
        augment=True,
        lr0=0.01,
        lrf=0.01,
        cos_lr=True,

        dropout=0.2,
        fliplr=0.5,
        flipud=0.5,

        degrees=15.0,
        perspective=0.0001,
        shear=2.0,

        mixup=0.15,
        copy_paste=0.1,

        multi_scale=True,

        workers=8,
        project=drive_save_path,
        name="train_model",
        device=0,
        deterministic=True,
        pretrained=True,
        exist_ok=True
    )

    val_results = model.val(data=dataset_output, split="val")

    test_results = model.val(data=dataset_output, split="test")

    best_model_path = "/content/drive/MyDrive/Train_YOLO/train_model/weights/best.pt"

    model = YOLO(best_model_path)

    test_dir = os.path.join(dataset_output, "test")
    class_names = sorted(os.listdir(test_dir))

    y_true = []
    y_pred = []

    for class_id, class_name in enumerate(class_names):
        class_path = os.path.join(test_dir, class_name)
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            result = model(img_path, verbose=False)
            pred_class = result[0].probs.top1

            y_true.append(class_id)
            y_pred.append(pred_class)

    print("\n📄 CLASSIFICATION REPORT:")
    report_str = classification_report(y_true, y_pred, target_names=class_names)
    print(report_str)

    # 2. Lưu Classification Report thành ảnh (dạng Heatmap)
    report_dict = classification_report(
        y_true, y_pred, target_names=class_names, output_dict=True
    )
    # Chuyển thành DataFrame và loại bỏ các dòng không cần thiết để vẽ biểu đồ đẹp hơn
    report_df = pd.DataFrame(report_dict).transpose()
    # Loại bỏ cột 'support' và dòng 'accuracy' để chỉ tập trung vào Precision, Recall, F1
    plot_df = report_df.drop(columns=["support"]).iloc[:-3, :]

    plt.figure(figsize=(10, 6))
    sns.heatmap(plot_df, annot=True, cmap="RdYlGn", fmt=".2f", cbar=True)
    plt.title("Classification Report Heatmap")
    plt.savefig(
        "classification_report_result.png", dpi=300, bbox_inches="tight"
    )  # Lưu ảnh chất lượng cao
    plt.show()

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )
    plt.xlabel("Dự đoán (Predicted)")
    plt.ylabel("Thực tế (Actual)")
    plt.title("Confusion Matrix - Trash Classification")
    plt.savefig("confusion_matrix_result.png")
    plt.show()

    elapsed_time = time.perf_counter() - start_time
    print(f"⏱️ Tổng thời gian: {format_duration(elapsed_time)}")

In [ ]:
from google.colab import drive
import os
import shutil
import time

drive.mount('/content/drive')

!pip install ultralytics split-folders

from ultralytics import YOLO

# Cấu hình đường dẫn
DRIVE_PATH = "/content/drive/MyDrive/Train_YOLO"
DATASET_ZIP = "/content/drive/MyDrive/Train_YOLO/dataset.zip" # Đường dẫn file zip trên Drive của bạn
LOCAL_DATASET_INPUT = "/content/dataset/input"
LOCAL_DATASET_OUTPUT = "/content/dataset/output"

last_checkpoint = os.path.join(DRIVE_PATH, "train_model", "weights", "last.pt")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
if not os.path.exists(LOCAL_DATASET_INPUT):
    !mkdir -p {LOCAL_DATASET_INPUT}
    !unzip -q {DATASET_ZIP} -d {LOCAL_DATASET_INPUT}


    import splitfolders
    if os.path.exists(LOCAL_DATASET_OUTPUT):
        shutil.rmtree(LOCAL_DATASET_OUTPUT)

    splitfolders.ratio(LOCAL_DATASET_INPUT, output=LOCAL_DATASET_OUTPUT, seed=1303, ratio=(0.70, 0.20, 0.10))
else:
    print("--- Dataset đã tồn tại trên máy ảo, bỏ qua bước giải nén. ---")

Copying files: 26717 files [00:26, 1016.65 files/s]


In [ ]:
start_time = time.perf_counter()

if os.path.exists(last_checkpoint):
    print("--- Đang thực hiện Resume thủ công... ---")
    # Load trọng số từ file last.pt
    model = YOLO(last_checkpoint)

    # results = model.train(
    #   data=LOCAL_DATASET_OUTPUT,
    #   epochs=120,
    #   resume=True
    # )

    # Train tiếp (không dùng resume=True)
    # results = model.train(
    #     data=LOCAL_DATASET_OUTPUT,
    #     epochs=120,
    #     imgsz=384,
    #     batch=64,
    #     lr0=2e-6,
    #     lrf=0.01,
    #     cos_lr=True,
    #     patience=15,
    #     dropout=0.1,
    #     fliplr=0.5,
    #     workers=8,
    #     optimizer="AdamW",
    #     project=DRIVE_PATH,
    #     name="train_model",
    #     exist_ok=True,
    #     device=0,
    #     deterministic=True,
    #     cache=False
    # )
    results = model.train(
        data=LOCAL_DATASET_OUTPUT,
        epochs=120,
        imgsz=384,
        batch=64,
        optimizer="SGD",
        momentum=0.937,
        patience=20,
        augment=True,
        lr0=0.01,
        lrf=0.01,
        cos_lr=True,

        dropout=0.2,
        fliplr=0.5,
        flipud=0.5,

        degrees=15.0,
        perspective=0.0001,
        shear=2.0,

        mixup=0.15,
        copy_paste=0.1,

        multi_scale=True,

        workers=8,
        project=DRIVE_PATH,
        name="train_model",
        device=0,
        deterministic=True,
        pretrained=True,
        exist_ok=True
    )

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from ultralytics import YOLO

# --- CẤU HÌNH ĐƯỜNG DẪN (Phải khớp với các cell trước) ---
DRIVE_SAVE_PATH = "/content/drive/MyDrive/Train_YOLO/optimized_yolov8_exp"
best_model_path = os.path.join(DRIVE_SAVE_PATH, "weights", "best.pt")
test_dir = "/content/dataset/output/test" # Đường dẫn tập test trên máy ảo

if os.path.exists(best_model_path):
    print(f"--- Đang tải model tốt nhất từ: {best_model_path} ---")
    model = YOLO(best_model_path)

    # 1. Lấy danh sách class
    class_names = sorted(os.listdir(test_dir))
    y_true = []
    y_pred = []

    print("--- Đang dự đoán trên tập Test (vui lòng đợi)... ---")
    for class_id, class_name in enumerate(class_names):
        class_path = os.path.join(test_dir, class_name)
        if not os.path.isdir(class_path): continue

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            try:
                result = model(img_path, verbose=False)
                pred_class = result[0].probs.top1
                y_true.append(class_id)
                y_pred.append(pred_class)
            except Exception as e:
                print(f"Lỗi khi xử lý ảnh {img_name}: {e}")

    # 2. In Classification Report
    print("\n📄 CLASSIFICATION REPORT:")
    report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    report_str = classification_report(y_true, y_pred, target_names=class_names)
    print(report_str)

    # Lưu report vào file text trên Drive
    with open(os.path.join(DRIVE_SAVE_PATH, "classification_report.txt"), "w") as f:
        f.write(report_str)

    # 3. Vẽ và lưu Heatmap Classification Report
    report_df = pd.DataFrame(report_dict).transpose()
    plot_df = report_df.drop(columns=["support"]).iloc[:-3, :]

    plt.figure(figsize=(12, 8))
    sns.heatmap(plot_df, annot=True, cmap="RdYlGn", fmt=".2f", cbar=True)
    plt.title("Classification Report Heatmap")

    # Lưu ảnh trực tiếp vào Drive
    heatmap_path = os.path.join(DRIVE_SAVE_PATH, "..", "classification_report_heatmap.png")
    plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
    print(f"✅ Đã lưu Heatmap tại: {heatmap_path}")
    plt.show()

    # 4. Vẽ và lưu Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names
    )
    plt.xlabel("Dự đoán (Predicted)")
    plt.ylabel("Thực tế (Actual)")
    plt.title("Confusion Matrix - Trash Classification")

    # Lưu ảnh trực tiếp vào Drive
    cm_path = os.path.join(DRIVE_SAVE_PATH, "..", "confusion_matrix_result.png")
    plt.savefig(cm_path, dpi=300, bbox_inches="tight")
    print(f"✅ Đã lưu Confusion Matrix tại: {cm_path}")
    plt.show()

    # Tính thời gian (nếu start_time tồn tại từ cell trước)
    try:
        elapsed_time = time.perf_counter() - start_time
        print(f"⏱️ Tổng thời gian huấn luyện và đánh giá: {format_duration(elapsed_time)}")
    except:
        print("⏱️ Đã hoàn thành đánh giá.")

else:
    print("❌ Không tìm thấy file best.pt trên Drive. Vui lòng kiểm tra lại quá trình train.")